In [49]:
!pip install mlflow


In [50]:
pip install dagshub

In [51]:
import dagshub
dagshub.init(repo_owner='balaram.sahu8', repo_name='YT-Sentiment-analysis', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Initialized MLflow to track repo "balaram.sahu8/YT-Sentiment-analysis"

Repository balaram.sahu8/YT-Sentiment-analysis initialized!

🏃 View run nervous-worm-118 at: https://dagshub.com/balaram.sahu8/YT-Sentiment-analysis.mlflow/#/experiments/1/runs/fb31103dba3347678b3853623772aa2b
🧪 View experiment at: https://dagshub.com/balaram.sahu8/YT-Sentiment-analysis.mlflow/#/experiments/1


In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [53]:
df = pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/refs/heads/main/data/reddit.csv')
df.head()

,clean_comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


In [54]:
df.isnull().sum()

,0
clean_comment,100
category,0


In [55]:
df[df['clean_comment'].isna()]

,clean_comment,category
413,NaN,0
605,NaN,0
2422,NaN,0
2877,NaN,0
3307,NaN,0
...,...,...
35975,NaN,0
36036,NaN,0
37043,NaN,0
37111,NaN,0


In [56]:
df.dropna(inplace=True)

In [57]:
df.isnull().sum()

,0
clean_comment,0
category,0


In [58]:
df.duplicated().sum()

np.int64(350)

In [59]:
df[df.duplicated()]

,clean_comment,category
375,,0
392,,0
617,aurum mom,0
651,,0
1222,,0
...,...,...
36915,who won,0
37044,,0
37125,hari,0
37158,top kek,1


In [60]:
df.drop_duplicates(inplace=True,keep='first')

In [61]:
df.shape

(36799, 2)

In [62]:
df.duplicated().sum()

np.int64(0)

In [63]:
pip install nltk

In [64]:
import re
import string
import nltk

from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download once
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")
nltk.download("stopwords")
nltk.download("omw-1.4")

# Initialize
lemmatizer = WordNetLemmatizer()

# Stopwords
stop_words = set(stopwords.words("english"))

# Words that should NOT be removed
keep_words = {
    "not", "no", "there", "but", "however", "yet"
}

stop_words = stop_words - keep_words


def preprocess_text(text):
    """
    Clean and preprocess text:
    1. Convert to lowercase
    2. Remove HTML tags
    3. Remove URLs
    4. Remove email addresses
    5. Remove numbers
    6. Remove punctuation and special characters
    7. Remove \\n and \\r
    8. Remove stopwords except selected words
    9. WordNet lemmatization
    """

   # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove \n and \r
    text = re.sub(r"[\n\r]+", " ", text)

    # 3. Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # 4. Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # 5. Remove email addresses
    text = re.sub(r"\S+@\S+", " ", text)

    # 6. Remove numbers
    text = re.sub(r"\d+", " ", text)

    # 7. Remove punctuation and special characters
    # Keep only alphabets and whitespace
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # 8. Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # 9. Tokenization
    words = word_tokenize(text)

    # 10. Remove stopwords
    words = [
        word for word in words
        if word not in stop_words
    ]

    # 11. WordNet lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    # 12. Join words back into sentence
    return " ".join(words)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [65]:
df['clean_comment'] = df['clean_comment'].apply(preprocess_text)

In [66]:
df['clean_comment'][0]

'family mormon never tried explain still stare puzzled time time like kind strange creature nonetheless come admire patience calmness equanimity acceptance compassion developed thing buddhism teach'

In [67]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [68]:
x = CountVectorizer(max_features=10000).fit_transform(df['clean_comment']).toarray()
y = df['category']

In [69]:
x.shape

(36799, 10000)

In [70]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [71]:
mlflow.set_tracking_uri("https://dagshub.com/balaram.sahu8/YT-Sentiment-analysis.mlflow")
mlflow.set_experiment("YT-Sentiment-analysis-RF-Basemodel")

# Re-import classification_report and confusion_matrix to ensure correct function reference
from sklearn.metrics import classification_report, confusion_matrix

with mlflow.start_run() as run:
    # Log a description for the run
    mlflow.set_tag("mlflow.runName", "RandomForest_Baseline_TrainTestSplit")
    mlflow.set_tag("experiment_type", "baseline")
    mlflow.set_tag("model_type", "RandomForestClassifier")

    # Add a description
    mlflow.set_tag("description", "Baseline RandomForest model for sentiment analysis using Bag of Words (BoW) with a simple train-test split")

    # Log parameters for the vectorizer
    mlflow.log_param("vectorizer_type", "CountVectorizer")
    mlflow.log_param("vectorizer_max_features", 10000)

    # Log Random Forest parameters
    n_estimators = 200
    max_depth = 15

    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)

    # Initialize and train the model
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(x_train, y_train)

    # Make predictions on the test set
    y_pred = model.predict(x_test)

    # Log metrics for each class and accuracy
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)

    # Get classification report as dictionary for individual metrics and as string for text log
    clf_report_dict = classification_report(y_test, y_pred, output_dict=True)
    clf_report_text = classification_report(y_test, y_pred)
    mlflow.log_text(clf_report_text, "classification_report.txt")

    for label, metrics in clf_report_dict.items():
        if isinstance(metrics, dict):  # For precision, recall, f1-score, etc.
            for metric, value in metrics.items():
                mlflow.log_metric(f"{label}_{metric}", value)

    # Confusion matrix plot
    conf_matrix_data = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix_data, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")

    # Save and log the confusion matrix plot
    plt.savefig("confusion_matrix.png")
    plt.close() # Close the plot to free up memory
    mlflow.log_artifact("/content/confusion_matrix.png")

    # Log the Random Forest model
    mlflow.sklearn.log_model(model, "random_forest_model")

    # Optionally log the dataset itself (if it's small enough)
    df.to_csv("dataset.csv", index=False)
    mlflow.log_artifact("/content/dataset.csv")

# Display final accuracy
print(f"Accuracy: {accuracy}")

2026/08/26 11:48:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForest_Baseline_TrainTestSplit at: https://dagshub.com/balaram.sahu8/YT-Sentiment-analysis.mlflow/#/experiments/1/runs/459dec4b15de4c5c97c310b0b6d1d4e2
🧪 View experiment at: https://dagshub.com/balaram.sahu8/YT-Sentiment-analysis.mlflow/#/experiments/1
Accuracy: 0.6501358695652174


In [72]:
clf_report = classification_report(y_test, y_pred)
print(clf_report)

              precision    recall  f1-score   support

          -1       1.00      0.02      0.03      1651
           0       0.68      0.82      0.74      2509
           1       0.63      0.85      0.72      3200

    accuracy                           0.65      7360
   macro avg       0.77      0.56      0.50      7360
weighted avg       0.73      0.65      0.57      7360



In [73]:
print(conf_matrix_data)

[[  26  494 1131]
 [   0 2048  461]
 [   0  489 2711]]


In [74]:
# if we will see here relacall is very low means out of 1651 data points only -1 classified 26 data points
# accuracy 1 beacuse there TP 26/26+0+0 = 1

# because of imbalane dataset issue we are getting low accuracy we have to do data balance

In [75]:
df.to_csv('cleand_df.csv',index=False)